In [52]:
import pandas as pd
import numpy as np

In [53]:
df=pd.read_csv("/content/cleaned_mental_health_data.csv")

In [54]:
df.columns

Index(['Age', 'Gender', 'Country', 'Education', 'Marital_Status',
       'Income_Level', 'Employment_Status', 'Work_Hours_Per_Week',
       'Remote_Work', 'Job_Satisfaction', 'Work_Stress_Level',
       'Work_Life_Balance', 'Ever_Bullied_At_Work',
       'Company_Mental_Health_Support', 'Exercise_Per_Week',
       'Sleep_Hours_Night', 'Caffeine_Drinks_Day', 'Alcohol_Frequency',
       'Smoking', 'Screen_Time_Hours_Day', 'Social_Media_Hours_Day',
       'Hobby_Time_Hours_Week', 'Diet_Quality', 'Financial_Stress',
       'Feeling_Sad_Down', 'Loss_Of_Interest', 'Sleep_Trouble', 'Fatigue',
       'Poor_Appetite_Or_Overeating', 'Feeling_Worthless',
       'Concentration_Difficulty', 'Anxious_Nervous', 'Panic_Attacks',
       'Mood_Swings', 'Irritability', 'Obsessive_Thoughts',
       'Compulsive_Behavior', 'Self_Harm_Thoughts', 'Suicidal_Thoughts',
       'Family_History_Mental_Illness', 'Previously_Diagnosed',
       'Ever_Sought_Treatment', 'On_Therapy_Now', 'On_Medication',
       'Traum

In [55]:
df=df.drop(['Country'] , axis=1)

In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 50 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            10000 non-null  int64  
 1   Gender                         10000 non-null  object 
 2   Education                      10000 non-null  object 
 3   Marital_Status                 10000 non-null  object 
 4   Income_Level                   10000 non-null  object 
 5   Employment_Status              10000 non-null  object 
 6   Work_Hours_Per_Week            10000 non-null  int64  
 7   Remote_Work                    10000 non-null  object 
 8   Job_Satisfaction               10000 non-null  int64  
 9   Work_Stress_Level              10000 non-null  int64  
 10  Work_Life_Balance              10000 non-null  int64  
 11  Ever_Bullied_At_Work           10000 non-null  int64  
 12  Company_Mental_Health_Support  10000 non-null  

In [58]:
df_fe = df.copy()

In [59]:
income_map = {
    'Low': 0,
    'Middle': 1,
    'High': 2
}

diet_map = {
    'Poor': 0,
    'Average': 1,
    'Good': 2,
    'Excellent': 3
}

alcohol_map = {
    'Never': 0,
    'Rarely': 1,
    'Sometimes': 2,
    'Weekly': 3,
    'Often': 4,
    'Daily': 5
}

discussion_map = {
    'Never': 0,
    'Rarely': 1,
    'Sometimes': 2,
    'Yes easily': 3
}

df_fe['Income_Level'] = df_fe['Income_Level'].map(income_map)

df_fe['Diet_Quality'] = df_fe['Diet_Quality'].map(diet_map)

df_fe['Alcohol_Frequency'] = df_fe['Alcohol_Frequency'].map(alcohol_map)

df_fe['Discuss_Mental_Health'] = df_fe[
    'Discuss_Mental_Health'
].map(discussion_map)

In [60]:
nominal_cols = [
    'Gender',
    'Education',
    'Marital_Status',
    'Employment_Status',
    'Remote_Work',
    'Company_Mental_Health_Support',
    'Smoking',
    'Exercise_Per_Week'
]

df_fe = pd.get_dummies(
    df_fe,
    columns=nominal_cols,
    drop_first=True
)

In [61]:
emotional_cols = [
    'Feeling_Sad_Down',
    'Feeling_Worthless',
    'Mood_Swings',
    'Anxious_Nervous',
    'Loneliness'
]

df_fe['Emotional_Distress_Score'] = df_fe[
    emotional_cols
].mean(axis=1)

In [62]:
cognitive_cols = [
    'Concentration_Difficulty',
    'Obsessive_Thoughts',
    'Compulsive_Behavior'
]

df_fe['Cognitive_Difficulty_Score'] = df_fe[
    cognitive_cols
].mean(axis=1)

In [63]:
stress_cols = [
    'Work_Stress_Level',
    'Financial_Stress'
]

df_fe['Stress_Burden_Score'] = df_fe[
    stress_cols
].mean(axis=1)

In [64]:
exhaustion_cols = [
    'Fatigue',
    'Sleep_Trouble',
    'Poor_Appetite_Or_Overeating'
]

df_fe['Exhaustion_Score'] = df_fe[
    exhaustion_cols
].mean(axis=1)

In [65]:
df_fe['Lifestyle_Imbalance'] = (
    df_fe['Screen_Time_Hours_Day']
    +
    df_fe['Social_Media_Hours_Day']
    -
    df_fe['Sleep_Hours_Night']
)

In [66]:
df_fe['Social_Isolation_Score'] = (
    df_fe['Loneliness']
    -
    df_fe['Social_Support']
)

In [67]:
df_fe['Stress_x_Sleep'] = (
    df_fe['Work_Stress_Level']
    *
    df_fe['Sleep_Hours_Night']
)

df_fe['Stress_x_Loneliness'] = (
    df_fe['Work_Stress_Level']
    *
    df_fe['Loneliness']
)

df_fe['SocialMedia_x_Sleep'] = (
    df_fe['Social_Media_Hours_Day']
    *
    df_fe['Sleep_Hours_Night']
)

In [68]:
X = df_fe.drop(
    'Has_Mental_Health_Issue',
    axis=1
)

y = df_fe['Has_Mental_Health_Issue']

In [69]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(
    X,
    y,
    random_state=42
)

mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
})

mi_df = mi_df.sort_values(
    by='MI_Score',
    ascending=False
)

mi_df.head(30)

,Feature,MI_Score
33,On_Therapy_Now,0.008827
62,Emotional_Distress_Score,0.006493
23,Panic_Attacks,0.005910
34,On_Medication,0.004973
39,Loneliness,0.004726
36,Social_Support,0.004637
61,Exercise_Per_Week_Never,0.004064
3,Job_Satisfaction,0.004057
35,Trauma_History,0.004020
38,Feel_Understood,0.003925


In [70]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

rf_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

rf_importance = rf_importance.sort_values(
    by='Importance',
    ascending=False
)

rf_importance.head(30)

,Feature,Importance
62,Emotional_Distress_Score,0.035017
68,Stress_x_Sleep,0.031694
70,SocialMedia_x_Sleep,0.031477
66,Lifestyle_Imbalance,0.031388
7,Sleep_Hours_Night,0.031129
10,Screen_Time_Hours_Day,0.030534
2,Work_Hours_Per_Week,0.030242
63,Cognitive_Difficulty_Score,0.029255
0,Age,0.028021
65,Exhaustion_Score,0.028016


In [72]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    rf,
    X,
    y,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_result.importances_mean
})

perm_df = perm_df.sort_values(
    by='Importance',
    ascending=False
)

perm_df.head(30)

,Feature,Importance
36,Social_Support,0.02171
63,Cognitive_Difficulty_Score,0.01714
62,Emotional_Distress_Score,0.01698
64,Stress_Burden_Score,0.01667
65,Exhaustion_Score,0.01575
69,Stress_x_Loneliness,0.01149
14,Financial_Stress,0.00521
15,Feeling_Sad_Down,0.00521
68,Stress_x_Sleep,0.00444
25,Irritability,0.00397


In [73]:
mi_selected = set(
    mi_df[
        mi_df['MI_Score'] > 0.01
    ]['Feature']
)

rf_selected = set(
    rf_importance[
        rf_importance['Importance'] > 0.005
    ]['Feature']
)

perm_selected = set(
    perm_df[
        perm_df['Importance'] > 0.001
    ]['Feature']
)

final_features = list(
    mi_selected |
    rf_selected |
    perm_selected
)

len(final_features)

42

In [74]:
X_selected = X[final_features]

X_selected.head()

,Mood_Swings,Stress_Burden_Score,Concentration_Difficulty,Caffeine_Drinks_Day,Fatigue,Stress_x_Loneliness,Social_Support,Anxious_Nervous,Exhaustion_Score,Emotional_Distress_Score,...,SocialMedia_x_Sleep,Work_Life_Balance,Feeling_Sad_Down,Gender_Male,Social_Media_Hours_Day,Work_Hours_Per_Week,Sleep_Hours_Night,Feel_Understood,Diet_Quality,Loss_Of_Interest
0,9,8.0,4,3,5,42,9,1,8.000000,5.6,...,29.76,7,8,True,4.8,27,6.2,4,1,3
1,5,5.0,6,3,9,7,3,8,5.666667,6.2,...,0.00,5,1,True,0.0,47,9.6,7,1,10
2,9,2.5,10,1,8,27,5,1,6.666667,6.4,...,23.10,8,4,True,3.0,53,7.7,6,2,7
3,0,2.0,5,1,10,15,1,3,7.333333,2.6,...,18.56,5,3,True,2.9,42,6.4,10,1,8
4,0,7.5,7,0,5,42,1,10,4.333333,5.4,...,30.80,7,6,False,4.0,13,7.7,1,3,8


In [77]:
import os

final_df = pd.concat(
    [X_selected, y],
    axis=1
)

# Create the directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

final_df.to_csv(
    '../data/processed/final_featured_dataset.csv',
    index=False
)